In [1]:
import os
import sys
import json
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path

from mpl_toolkits.axes_grid1.inset_locator import mark_inset
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import matplotlib.patches as patches

from scipy.signal import savgol_filter, find_peaks
from scipy.interpolate import UnivariateSpline
import scipy as scipy
from scipy import optimize
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from scipy.signal import savgol_filter, find_peaks
import matplotlib.pyplot as plt
import latex

import orpl.baseline_removal as rt
from orpl.cosmic_ray import crfilter_single,crfilter_multi

from matplotlib import rc
# rc('font',**{'family':'sans-serif','sans-serif':['Helvetica']})
# for Palatino and other serif fonts use:
rc('font',**{'family':'serif','serif':['Palatino']})
rc('text', usetex=True)


In [15]:
def MakeDF(data_df):
    data = []
    for idx,df in data_df.iterrows():
        info = {}
        raw = np.array(df.Raw).transpose()
        bkg = np.array(df.Background)
        y1 = np.mean(raw, axis=0)-bkg[130:]
        Raman, AF = rt.bubblefill(savgol_filter(y1,11,3), min_bubble_widths=40)
        Raman_raw, AF_raw = rt.bubblefill(y1, min_bubble_widths=40)
        mw = df.Power
        s = df.ExpTime*df.SpecPerAcqui/1000
        max_raw = np.argmax(raw[0,130:])+130
        info['DynamicRange'] = raw[0,max_raw]/65535
        info['DynamicRangeLast'] = raw[-1,max_raw]/65535
        info['PhotoBleaching'] = raw[:,max_raw]
        info['mWs'] = mw/1000*s
        info['Power'] = mw/1000
        info['ExpTime'] = df.ExpTime
        info['NbSpectra'] = df.SpecPerAcqui
        info['IntegratedTime'] = s
        info['QF'] = GetQF(Raman)
        info['SBR'] = np.sum(Raman)/np.sum(AF)
        info['SBR_raw'] = np.sum(Raman_raw)/np.sum(AF_raw)
        info['Photon'] = np.sum(y1)/(mw*s)
        info['xaxis'] = df.xaxis
        info['SpecSBR'] = Raman/AF
        info['SpecSBR_raw'] = Raman_raw/AF_raw
        info['Raman'] = Raman/(mw*s)
        info['Raman_raw'] = Raman_raw/(mw*s)
        info['SNV'] = SNV(Raman)
        info['SNV_raw'] = SNV(Raman_raw)

        data.append(info)
    return pd.DataFrame.from_dict(data)


In [17]:
def SNV(x):
    if x.ndim == 1:
        return (x - np.mean(x)) / np.std(x)
    elif x.ndim == 2:
        return (x - np.mean(x, axis=1)[:, np.newaxis]) / np.std(x, axis=1)[:, np.newaxis]

def QualityFactor(raman):
    raman_ = SNV(raman)
    deviation_sign = np.sign(raman_)
    deviation2 = (raman_ - raman_.mean()) ** 2
    quality_factor = (deviation_sign * deviation2).mean()
    return quality_factor

def GetCorrectionCurve(x, nist):
    A0 = 9.71937e-02
    A1 = 2.28325e-04
    A2 = -5.86762e-08
    A3 = 2.16023e-10
    A4 = -9.77171e-14
    A5 = 1.15596e-17
    
    nist_theo = A0 + A1*x + A2*x**2 + A3*x**3 + A4*x**4 + A5*x**5
    curve = nist / nist_theo
    return curve / np.amax(curve)

In [ ]:
# --- Power calibration dicts ---
laser_calibration_current = np.array([640, 650, 660])
laser_calibration_power = np.array([26.2, 27.5, 29.0])  

# Spline interpolation function
from scipy.interpolate import interp1d
laser_current_to_power = interp1d(
    laser_calibration_current, 
    laser_calibration_power, 
    kind='linear', 
    fill_value='extrapolate'
)


In [19]:
# --- Load joblib files from folder ---
def load_joblib_files(path_to_data):
    joblib_files = []
    path = Path(path_to_data)
    for file in sorted(path.glob('**/*.joblib')):  # recursive search
        try:
            data = joblib.load(file)
            joblib_files.append({'data': data, 'path': str(file)})
        except Exception as e:
            print(f"Failed to load {file}: {e}")
    return joblib_files

In [23]:
def extract_calibration(tylenol_files, nist_files):
    if not tylenol_files:
        raise ValueError("No Tylenol calibration files found.")
    if not nist_files:
        raise ValueError("No NIST calibration files found.")

    tylenol_shift = np.array([651.6, 797.2, 857.9, 1168.5, 1236.8,
                              1278.5, 1323.9, 1371.5, 1609, 1648.4])
    info = {}

    # --- Tylenol for x-axis calibration
    for obj in tylenol_files:
        data = obj['data']
        accum = np.array(data['accumulations'])
        background = np.array(data['background'])
        xaxis = np.array(data['xaxis'])

        mean_spec = accum.mean(axis=0) - background.squeeze()
        mbw = int(mean_spec.shape[0] / 20.48)
        tyl, _ = rt.bubblefill(mean_spec, min_bubble_widths=mbw)
        snv = SNV(tyl) - np.min(SNV(tyl))
        peaks, _ = find_peaks(snv, height=1.5)
        peaks = peaks[:10]  # limit to first 10
        raman_shift = np.polyval(np.polyfit(peaks, tylenol_shift, 2), np.arange(snv.size))

        info['xaxis'] = raman_shift
        info['Tylenol'] = snv
        info['Tyl'] = tyl
        break  # only take the first

    # --- NIST for intensity correction
    for obj in nist_files:
        data = obj['data']
        accum = np.array(data['accumulations'])
        nist = accum.mean(axis=0)
        info['IRC'] = GetCorrectionCurve(info['xaxis'], nist)
        info['NIST'] = nist
        break  # only take the first

    print("Calibration dict keys:", info.keys())
    return info



In [21]:
# --- Process full joblib data ---
def process_data(joblib_files, calib_df):
    full_data = []
    for obj in joblib_files:
        data = obj['data']
        comment = data['metadata'].get('comment', '')
        calib = calib_df # assuming one calibration
        x = calib['xaxis']
        idx_1 = np.where(x <= 600)[0][-1]
        idx_2 = 1023
        raw = np.array(data['accumulations'])[:, idx_1:idx_2]
        bkg = np.array(data['background'])[:, idx_1:idx_2].squeeze()
        raw = crfilter_multi(raw)
        bkg = crfilter_single(bkg)
        rawfinal = raw - bkg
        lp_current = data['acquisition_profile']['laser_current']
        lp = float(laser_current_to_power(lp_current))  # interpolated power in mW


        nb = data['acquisition_profile']['n_accumulations']
        et = data['acquisition_profile']['exposure_time']
        

        try:
            corr = rawfinal.mean(axis=0) / calib['IRC'][idx_1:idx_2]
        except:
            continue

        mbw = int(corr.shape[0] / 20.48)
        Raman, AF = rt.bubblefill(corr, min_bubble_widths=mbw)
        norm_factor = nb * et * lp / 1000

        info = {
            'Measure': comment,
            'xaxis': x[idx_1:idx_2],
            'Raw': rawfinal.mean(axis=0),
            'RawNormalized': corr / norm_factor,
            'Raman': Raman,
            'RamanFilt': savgol_filter(Raman, 11, 3),
            'RamanNormalized': Raman / norm_factor,
            'RamanFiltNormalized': savgol_filter(Raman, 11, 3) / norm_factor,
            'RamanSNV': SNV(Raman),
            'RamanFiltSNV': SNV(savgol_filter(Raman, 11, 3)),
            'NbSpectra': nb,
            'ExposureTime': et,
            'LaserPower': lp
        }
        full_data.append(info)
    return pd.DataFrame.from_dict(full_data)


In [26]:
# Define paths
tylenol_path = r"C:\Users\tommy\Desktop\Maitrîse\Experiments\Fred_background\exp_background_1804\new-probe\tylenol_0.joblib"
nist_path = r"C:\Users\tommy\Desktop\Maitrîse\Experiments\Fred_background\exp_background_1804\new-probe\nist_0.joblib"
data_path = r"C:\Users\tommy\Desktop\Maitrîse\Experiments\Fred_background\exp_background_1804\new-probe\acquisitions"

# Load files
tylenol_files = [{'data': joblib.load(tylenol_path), 'path': tylenol_path}]
nist_files = [{'data': joblib.load(nist_path), 'path': nist_path}]
joblib_files = load_joblib_files(data_path)

# Extract calibration from Tylenol and NIST
calib_df = extract_calibration(tylenol_files, nist_files)

# Process data
full_df = process_data(joblib_files, calib_df)

# Save processed data
full_df.to_pickle(r"C:\Users\tommy\Desktop\Maitrîse\Experiments\Fred_background\processed_data")

print("Processing complete. DataFrame saved.")


Calibration dict keys: dict_keys(['xaxis', 'Tylenol', 'Tyl', 'IRC', 'NIST'])


ValueError: zero-size array to reduction operation maximum which has no identity

In [ ]:
print(full_df)

In [ ]:
import matplotlib.pyplot as plt


plt.figure(figsize=(12, 4))
plt.plot(calib_df.iloc[0]['xaxis'], calib_df.iloc[0]['Tylenol'], label="Tylenol SNV", color='black')
plt.title("Tylenol Calibration Spectrum (SNV)")
plt.xlabel("Raman Shift (cm⁻¹)")
plt.ylabel("Intensity (a.u.)")
plt.grid(True)
plt.legend()
plt.show()


plt.figure(figsize=(12, 4))
plt.plot(calib_df.iloc[0]['xaxis'], calib_df.iloc[0]['IRC'], label="Instrument Response Correction (NIST)", color='blue')
plt.title("Instrument Response Correction Curve")
plt.xlabel("Raman Shift (cm⁻¹)")
plt.ylabel("Correction Factor")
plt.grid(True)
plt.legend()
plt.show()


In [ ]:

avg_raman = np.vstack(full_df['RamanNormalized']).mean(axis=0)
xaxis = full_df.iloc[0]['xaxis']

plt.figure(figsize=(12, 4))
plt.plot(xaxis, avg_raman, label="Average Raman (Normalized)", color='green')
plt.title("Average Raman Spectrum (Normalized by mW·s)")
plt.xlabel("Raman Shift (cm⁻¹)")
plt.ylabel("Intensity (a.u.)")
plt.grid(True)
plt.legend()
plt.show()
